<a href="https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I frame my Content Refresh / Content Opportunity Scoring lane as a binary classification problem. The task is to estimate whether a page is likely to be in a declining search-performance state using observable page and search-performance signals. The positive class represents a declining page, while the negative class represents a page that is not declining. The output would support a content team's decision about which pages should receive refresh review first. Classification is useful here because the decision can be represented as declining versus not declining while combining multiple signals instead of relying on one fixed rule.


In [ ]:
# This cell is for CODE
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/AbdulWasay65/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Working directory:", os.getcwd())
print("Total rows:", len(df))
print("Declining:", df["is_declining_label"].sum())
print("Not declining:", (df["is_declining_label"] == 0).sum())
print("Declining rate:", df["is_declining_label"].mean().round(3))

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Working directory: /content/flyrank-ml-internship
Total rows: 30000
Declining: 16262
Not declining: 13738
Declining rate: 0.542


## 2. Target or proxy

My target is whether a page is in a declining search-performance state. I define the binary target as `is_declining_label = 1` when `trend_direction` is `down`, and `0` otherwise. This label comes from an observed trend classification already present in the dataset rather than a manually chosen threshold. The target is therefore a proxy for pages that may warrant closer review for content refresh.


In [ ]:
# This cell is for CODE
print("Target column:", "is_declining_label")
print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean().round(3))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target column: is_declining_label

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.542


## 3. Success metric

I would use recall as the primary success metric because the goal is to identify pages that are likely to be declining and may deserve refresh review. A false negative means missing a declining page, which could cause a useful refresh opportunity to be overlooked. Higher recall means the model identifies a larger share of the pages that are actually declining. I would use precision as a secondary metric to monitor how many flagged pages are actually declining.


In [ ]:
# This cell is for CODE
from sklearn.metrics import recall_score

y_true = df["is_declining_label"]

# Baseline: predict every page as declining
y_baseline = [1] * len(df)

baseline_recall = recall_score(y_true, y_baseline)

print("Baseline recall:", baseline_recall)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Baseline recall: 1.0


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content record for a client. Each row represents one piece of content with its content characteristics and observed search-performance measurements. The dataset includes content-level attributes such as word count and content age, along with search-performance signals such as impressions, clicks, CTR, and average position. For this task, the target label indicates whether that content record is classified as declining.


In [ ]:
# This cell is for CODE
columns_to_show = [
    "content_id",
    "client_id",
    "content_type",
    "word_count",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "is_declining_label"
]

unit_df = df[columns_to_show].head(5)

display(unit_df)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,content_id,client_id,content_type,word_count,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,187,3803,29,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,445,15320,7,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,141,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,463,11751,58,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,263,19140,24,0.13,44.0,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could flag pages using one threshold, such as pages older than a certain number of days or pages with impressions below a chosen value. However, content performance can depend on multiple signals at the same time, including content age, impressions, CTR, average position, word count, and recent performance. A fixed rule would require manually choosing thresholds and may miss combinations of signals that are associated with declining performance. ML is useful here because a classification model can learn patterns across multiple features and produce a probability or class prediction that can support refresh-review prioritization. This does not mean ML is automatically better; its value would need to be demonstrated against a simple baseline using held-out data.


In [ ]:
# This cell is for CODE
feature_candidates = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("Candidate features:")
for feature in feature_candidates:
    print("-", feature)

print("\nTarget:")
print("is_declining_label")

print("\nFeatures excluded to avoid target leakage:")
print("- trend_direction")
print("- trend_pct")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Candidate features:
- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr
- word_count

Target:
is_declining_label

Features excluded to avoid target leakage:
- trend_direction
- trend_pct


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.